#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, when, count, to_date, length, mean, right
from pyspark.sql import Window

In [0]:
#Função Info()

# Info dos Dados
# Tipo da Coluna
# Quantidade de linhas
# Quantidade de nulos
# Quantidade de Valores únicos

def info(x):

    # Número total de linhas
    n_rows = x.count()

    summary = []
    for c in x.columns:
        dtype = dict(x.dtypes)[c]
        n_nulls = x.filter(F.col(c).isNull()).count()
        n_uniques = x.select(c).distinct().count()
        summary.append((c, dtype, n_rows, n_nulls, n_uniques))

    # Crie o DataFrame de resumo
    summary_df = spark.createDataFrame(
        summary,
        ["coluna", "tipo", "qtd_linhas", "qtd_nulos", "qtd_valores_unicos"]
    )

    summary_df.show()
    
    return

#Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.px_cat_g1v2")

In [0]:
info(df) 
df.show(n=5)

#Transformations

## Rename columns' names

In [0]:
RENAME_MAP = {
    "ID": "Id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance"
   
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimm

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Unique Values

In [0]:
info(df)
print("", "Não há duplicações")


## Normalization

### Gender

In [0]:

df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)


#Write into Silver Layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_product_category")
